In [1]:
from sunpy.net import Fido, attrs as a

# ============================================================
# 1. INTERVALO TEMPORAL
# ============================================================

# 24 horas completas del 3 de julio de 2024
time = a.Time(
    "2024-07-03 00:00:00",
    "2024-07-03 23:59:30"
)

# ============================================================
# 2. SERIE HMI
# ============================================================

# Dopplergramas HMI con cadencia de 45 segundos
series = a.jsoc.Series("hmi.V_45s")


# ============================================================
# 3. SEGMENTO
# ============================================================

# Solo necesitamos el Dopplergrama
segment = a.jsoc.Segment("Dopplergram")


# ============================================================
# 4. EMAIL REGISTRADO EN JSOC
# ============================================================

email = a.jsoc.Notify("darodriguezto@unal.edu.co")


# ============================================================
# 5. BUSCAR LOS DATOS
# ============================================================

result = Fido.search(
    time,
    series,
    segment,
    email
)

print(result)

/home/daniel/miniforge3/envs/dkist/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Results from 1 Provider:

1920 Results from the JSOCClient:
Source: http://jsoc.stanford.edu

         T_REC          TELESCOP  INSTRUME  WAVELNTH CAR_ROT
----------------------- -------- ---------- -------- -------
2024.07.03_00:00:45_TAI  SDO/HMI HMI_FRONT2   6173.0  2286.0
2024.07.03_00:01:30_TAI  SDO/HMI HMI_FRONT2   6173.0  2286.0
2024.07.03_00:02:15_TAI  SDO/HMI HMI_FRONT2   6173.0  2286.0
2024.07.03_00:03:00_TAI  SDO/HMI HMI_FRONT2   6173.0  2286.0
2024.07.03_00:03:45_TAI  SDO/HMI HMI_FRONT2   6173.0  2286.0
2024.07.03_00:04:30_TAI  SDO/HMI HMI_FRONT2   6173.0  2286.0
2024.07.03_00:05:15_TAI  SDO/HMI HMI_FRONT2   6173.0  2286.0
2024.07.03_00:06:00_TAI  SDO/HMI HMI_FRONT2   6173.0  2286.0
2024.07.03_00:06:45_TAI  SDO/HMI HMI_FRONT2   6173.0  2286.0
2024.07.03_00:07:30_TAI  SDO/HMI HMI_FRONT2   6173.0  2286.0
2024.07.03_00:08:15_TAI  SDO/HMI HMI_FRONT2   6173.0  2286.0
2024.07.03_00:09:00_TAI  SDO/HMI HMI_FRONT2   6173.0  2286.0
2024.07.03_00:09:45_TAI  SDO/HMI HMI_FRONT2   6173.0

In [ ]:
# ============================================================
# 6. RUTA DE DESCARGA
# ============================================================

path = "../data/2024_07_03/dopplergrams/{file}"

# ============================================================
# 7. DESCARGA
# ============================================================

downloaded_files = Fido.fetch(
    result,
    path=path
)

print(downloaded_files)

In [2]:
from sunpy.net import Fido, attrs as a
from pathlib import Path
from datetime import datetime, timedelta
import gc
import time

# ============================================================
# CONFIGURACIÓN
# ============================================================

output_dir = Path("../data/2024_07_03/dopplergrams/")
output_dir.mkdir(parents=True, exist_ok=True)

email = a.jsoc.Notify("darodriguezto@unal.edu.co")

series = a.jsoc.Series("hmi.V_45s")
segment = a.jsoc.Segment("Dopplergram")

start_day = datetime(2024, 7, 3, 0, 0, 0)


# ============================================================
# DESCARGA POR BLOQUES DE UNA HORA
# ============================================================

for hour in range(24):

    start = start_day + timedelta(hours=hour)

    # Restamos 1 segundo para no incluir el primer registro
    # de la hora siguiente
    end = start + timedelta(hours=1) - timedelta(seconds=1)

    print("\n" + "=" * 60)
    print(f"Bloque {hour + 1:02d}/24")
    print(f"{start}  -->  {end}")
    print("=" * 60)

    # --------------------------------------------------------
    # 1. Buscar solamente esta hora
    # --------------------------------------------------------

    result = Fido.search(
        a.Time(start, end),
        series,
        segment,
        email
    )

    n_records = len(result[0])

    print(f"Registros encontrados en JSOC: {n_records}")

    # Para una hora completa esperamos normalmente 80:
    # 3600 / 45 = 80

    # --------------------------------------------------------
    # 2. Descargar este pequeño bloque
    # --------------------------------------------------------

    downloaded = Fido.fetch(
        result,
        path=str(output_dir / "{file}")
    )

    print(f"Bloque {hour:02d}:00 terminado.")

    # --------------------------------------------------------
    # 3. Estado actual
    # --------------------------------------------------------

    n_local = len(list(output_dir.glob("*.fits")))

    print(f"FITS actualmente en disco: {n_local} / 1920")

    # --------------------------------------------------------
    # 4. Liberar objetos antes de continuar
    # --------------------------------------------------------

    del result
    del downloaded

    gc.collect()

    # Pequeña pausa entre solicitudes
    time.sleep(2)


print("\n" + "=" * 60)
print("DESCARGA POR BLOQUES FINALIZADA")
print("=" * 60)

files = list(output_dir.glob("*.fits"))

print(f"Total de FITS encontrados: {len(files)}")
print(f"Esperados: 1920")


Bloque 01/24
2024-07-03 00:00:00  -->  2024-07-03 00:59:59
Registros encontrados en JSOC: 81


2026-09-24 13:12:38 - drms - INFO: Export request pending. [id=JSOC_20260924_004632, status=2]
2026-09-24 13:12:38 - drms - INFO: Waiting for 0 seconds...
2026-09-24 13:12:40 - drms - INFO: Export request pending. [id=JSOC_20260924_004632, status=1]
2026-09-24 13:12:40 - drms - INFO: Waiting for 5 seconds...
2026-09-24 13:12:46 - drms - INFO: Export request pending. [id=JSOC_20260924_004632, status=1]
2026-09-24 13:12:46 - drms - INFO: Waiting for 5 seconds...
2026-09-24 13:12:52 - drms - INFO: Export request pending. [id=JSOC_20260924_004632, status=1]
2026-09-24 13:12:52 - drms - INFO: Waiting for 5 seconds...
2026-09-24 13:12:58 - drms - INFO: Export request pending. [id=JSOC_20260924_004632, status=1]
2026-09-24 13:12:58 - drms - INFO: Waiting for 5 seconds...
2026-09-24 13:13:04 - drms - INFO: Export request pending. [id=JSOC_20260924_004632, status=1]
2026-09-24 13:13:04 - drms - INFO: Waiting for 5 seconds...
2026-09-24 13:13:21 - drms - INFO: Export request pending. [id=JSOC_20

INFO: 81 URLs found for download. Full request totaling 1360MB [sunpy.net.jsoc.jsoc]


Files Downloaded:  78%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                                       | 63/81 [08:46<02:18,  7.68s/file]2026-09-24 13:23:04 - parfive - INFO: http://jsoc.stanford.edu/SUM2/D2041038303/S00000/hmi.v_45s.20240703_004800_TAI.2.Dopplergram.fits failed to download with exception

Files Downloaded:  78%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                                       | 63/81 [08:46<02:30,  8.36s/file]


1/0 files failed to download. Please check `.errors` for details
Bloque 00:00 terminado.
FITS actualmente en disco: 681 / 1920

Bloque 02/24
2024-07-03 01:00:00  -->  2024-07-03 01:59:59


KeyboardInterrupt: 

In [6]:
from sunpy.net import Fido, attrs as a
from pathlib import Path
from datetime import datetime, timedelta
import gc
import time

output_dir = Path("../data/2024_07_03/dopplergrams/")
email = a.jsoc.Notify("darodriguezto@unal.edu.co")

series = a.jsoc.Series("hmi.V_45s")
segment = a.jsoc.Segment("Dopplergram")

# ============================================================
# CONTINUAR DESDE DONDE QUEDÓ LA DESCARGA
# ============================================================

start_day = datetime(2024, 7, 3, 20, 30)
end_day   = datetime(2024, 7, 4, 0, 0)

current = start_day

while current < end_day:

    end = min(current + timedelta(hours=1), end_day)

    print("\n" + "=" * 60)
    print(f"{current} --> {end}")
    print("=" * 60)

    result = Fido.search(
        a.Time(current, end - timedelta(seconds=1)),
        series,
        segment,
        email
    )

    print(f"Registros encontrados: {len(result[0])}")

    downloaded = Fido.fetch(
        result,
        path=str(output_dir / "{file}")
    )

    n_local = len(list(output_dir.glob("*.fits")))
    print(f"FITS actualmente en disco: {n_local}")

    del result
    del downloaded
    gc.collect()

    current = end
    time.sleep(2)

print("\nDESCARGA FINALIZADA")


2024-07-03 20:30:00 --> 2024-07-03 21:30:00
Registros encontrados: 81


2026-09-24 22:29:36 - drms - INFO: Export request pending. [id=JSOC_20260925_000940, status=2]
2026-09-24 22:29:36 - drms - INFO: Waiting for 0 seconds...
2026-09-24 22:29:36 - drms - INFO: Export request pending. [id=JSOC_20260925_000940, status=1]
2026-09-24 22:29:36 - drms - INFO: Waiting for 5 seconds...
2026-09-24 22:29:42 - drms - INFO: Export request pending. [id=JSOC_20260925_000940, status=1]
2026-09-24 22:29:42 - drms - INFO: Waiting for 5 seconds...
2026-09-24 22:29:47 - drms - INFO: Export request pending. [id=JSOC_20260925_000940, status=1]
2026-09-24 22:29:47 - drms - INFO: Waiting for 5 seconds...
2026-09-24 22:29:53 - drms - INFO: Export request pending. [id=JSOC_20260925_000940, status=1]
2026-09-24 22:29:53 - drms - INFO: Waiting for 5 seconds...
2026-09-24 22:29:58 - drms - INFO: Export request pending. [id=JSOC_20260925_000940, status=1]
2026-09-24 22:29:58 - drms - INFO: Waiting for 5 seconds...
2026-09-24 22:30:05 - drms - INFO: Export request pending. [id=JSOC_20

INFO: 81 URLs found for download. Full request totaling 1358MB [sunpy.net.jsoc.jsoc]


Files Downloaded:   1%|██▏                                                                                                                                                                                 | 1/81 [00:01<02:06,  1.58s/file]
hmi.v_45s.20240703_203130_TAI.2.Dopplergram.fits:   0%|                                                                                                                                                         | 0.00/17.6M [00:00<?, ?B/s]
hmi.v_45s.20240703_203130_TAI.2.Dopplergram.fits:   0%|                                                                                                                                              | 1.02k/17.6M [00:00<4:32:33, 1.07kB/s]
hmi.v_45s.20240703_203130_TAI.2.Dopplergram.fits:   0%|▎                                                                                                                                               | 33.4k/17.6M [00:01<07:08, 41.0kB/s]
hmi.v_45s.20240703_203130_TAI.2.Dopplergram.fits:   

FITS actualmente en disco: 1706

2024-07-03 21:30:00 --> 2024-07-03 22:30:00
Registros encontrados: 81


...

FITS actualmente en disco: 1866

2024-07-03 23:30:00 --> 2024-07-04 00:00:00
Registros encontrados: 41


2026-09-24 23:17:26 - drms - INFO: Export request pending. [id=JSOC_20260925_001151, status=2]
2026-09-24 23:17:26 - drms - INFO: Waiting for 0 seconds...
2026-09-24 23:17:26 - drms - INFO: Export request pending. [id=JSOC_20260925_001151, status=1]
2026-09-24 23:17:26 - drms - INFO: Waiting for 5 seconds...
2026-09-24 23:17:32 - drms - INFO: Export request pending. [id=JSOC_20260925_001151, status=1]
2026-09-24 23:17:32 - drms - INFO: Waiting for 5 seconds...
2026-09-24 23:17:37 - drms - INFO: Export request pending. [id=JSOC_20260925_001151, status=1]
2026-09-24 23:17:37 - drms - INFO: Waiting for 5 seconds...
2026-09-24 23:17:43 - drms - INFO: Export request pending. [id=JSOC_20260925_001151, status=1]
2026-09-24 23:17:43 - drms - INFO: Waiting for 5 seconds...
2026-09-24 23:17:49 - drms - INFO: Export request pending. [id=JSOC_20260925_001151, status=1]
2026-09-24 23:17:49 - drms - INFO: Waiting for 5 seconds...
2026-09-24 23:17:54 - drms - INFO: Export request pending. [id=JSOC_20

INFO: 41 URLs found for download. Full request totaling 688MB [sunpy.net.jsoc.jsoc]


Files Downloaded:   2%|████▍                                                                                                                                                                               | 1/41 [00:01<00:56,  1.40s/file]
hmi.v_45s.20240703_233130_TAI.2.Dopplergram.fits:   0%|                                                                                                                                                         | 0.00/17.6M [00:00<?, ?B/s]
hmi.v_45s.20240703_233130_TAI.2.Dopplergram.fits:   0%|                                                                                                                                              | 1.02k/17.6M [00:00<4:33:38, 1.07kB/s]
hmi.v_45s.20240703_233130_TAI.2.Dopplergram.fits:   0%|▎                                                                                                                                               | 33.4k/17.6M [00:01<07:08, 41.0kB/s]
hmi.v_45s.20240703_233130_TAI.2.Dopplergram.fits:   

FITS actualmente en disco: 1906

DESCARGA FINALIZADA


In [1]:
from astropy.io import fits
from pathlib import Path

filename = Path(
    "../data/2024_07_03/dopplergrams/"
    "hmi.v_45s.20240703_120000_TAI.2.Dopplergram.fits"
)

# Archivo temporal
tmp_file = filename.with_suffix(".tmp.fits")

# ============================================================
# LEER DOPPLERGRAMA DE HDU 1
# ============================================================

with fits.open(filename, memmap=True) as hdul:

    print("Antes:")
    print(hdul.info())

    data = hdul[1].data
    header = hdul[1].header.copy()

    # Crear FITS nuevo con los datos directamente en HDU 0
    primary = fits.PrimaryHDU(
        data=data,
        header=header
    )

    primary.writeto(
        tmp_file,
        overwrite=True
    )

# ============================================================
# VERIFICAR ARCHIVO NUEVO
# ============================================================

with fits.open(tmp_file) as hdul:

    assert hdul[0].data is not None
    assert hdul[0].header["NAXIS"] == 2
    assert hdul[0].data.shape == (4096, 4096)

    print("\nDespués:")
    print(hdul.info())

# ============================================================
# REEMPLAZAR ORIGINAL SOLO SI TODO SALIÓ BIEN
# ============================================================

tmp_file.replace(filename)

print("\nConversión terminada:")
print(filename)

Antes:
Filename: ../data/2024_07_03/dopplergrams/hmi.v_45s.20240703_120000_TAI.2.Dopplergram.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       6   ()      
  1  COMPRESSED_IMAGE    1 CompImageHDU    111   (4096, 4096)   int16 (rescales to float32)   
None

Después:
Filename: ../data/2024_07_03/dopplergrams/hmi.v_45s.20240703_120000_TAI.2.Dopplergram.tmp.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     109   (4096, 4096)   float32   
None

Conversión terminada:
../data/2024_07_03/dopplergrams/hmi.v_45s.20240703_120000_TAI.2.Dopplergram.fits


In [2]:
from astropy.io import fits
from pathlib import Path

data_dir = Path("../data/2024_07_03/dopplergrams")

files = sorted(data_dir.glob("*.Dopplergram.fits"))

print(f"Archivos encontrados: {len(files)}")

converted = 0
already_ok = 0
failed = []

for i, filename in enumerate(files, start=1):

    print(f"[{i}/{len(files)}] {filename.name}")

    tmp_file = filename.with_suffix(".tmp.fits")

    try:

        # ====================================================
        # INSPECCIONAR ARCHIVO
        # ====================================================

        with fits.open(filename, memmap=True) as hdul:

            # Si HDU 0 ya contiene una imagen, no tocarlo
            if (
                hdul[0].data is not None
                and hdul[0].header.get("NAXIS", 0) == 2
            ):
                print("    Ya está en formato correcto.")
                already_ok += 1
                continue

            # Comprobar que existe HDU 1 con imagen
            if len(hdul) < 2 or hdul[1].data is None:
                raise RuntimeError(
                    "No se encontró imagen válida en HDU 1."
                )

            data = hdul[1].data
            header = hdul[1].header.copy()

            # Crear nuevo FITS
            fits.PrimaryHDU(
                data=data,
                header=header
            ).writeto(
                tmp_file,
                overwrite=True
            )

        # ====================================================
        # VERIFICAR TEMPORAL
        # ====================================================

        with fits.open(tmp_file, memmap=True) as hdul_tmp:

            if hdul_tmp[0].data is None:
                raise RuntimeError(
                    "El FITS convertido no contiene datos."
                )

            if hdul_tmp[0].header.get("NAXIS") != 2:
                raise RuntimeError(
                    "NAXIS del FITS convertido no es 2."
                )

            if hdul_tmp[0].data.shape != (4096, 4096):
                raise RuntimeError(
                    f"Shape inesperado: {hdul_tmp[0].data.shape}"
                )

        # ====================================================
        # REEMPLAZAR ORIGINAL
        # ====================================================

        tmp_file.replace(filename)

        converted += 1
        print("    OK")

    except Exception as e:

        print(f"    ERROR: {e}")
        failed.append(filename.name)

        # Eliminar temporal incompleto
        if tmp_file.exists():
            tmp_file.unlink()


# ============================================================
# RESUMEN
# ============================================================

print("\n========================================")
print("CONVERSIÓN TERMINADA")
print("========================================")

print(f"Convertidos:       {converted}")
print(f"Ya correctos:      {already_ok}")
print(f"Errores:           {len(failed)}")

if failed:
    print("\nArchivos con error:")
    for f in failed:
        print(f"  {f}")

Archivos encontrados: 1906
[1/1906] hmi.v_45s.20240703_000045_TAI.2.Dopplergram.fits
    OK
[2/1906] hmi.v_45s.20240703_000130_TAI.2.Dopplergram.fits
    OK
[3/1906] hmi.v_45s.20240703_000215_TAI.2.Dopplergram.fits
    OK
[4/1906] hmi.v_45s.20240703_000300_TAI.2.Dopplergram.fits
    OK
[5/1906] hmi.v_45s.20240703_000345_TAI.2.Dopplergram.fits
    OK
[6/1906] hmi.v_45s.20240703_000430_TAI.2.Dopplergram.fits
    OK
[7/1906] hmi.v_45s.20240703_000515_TAI.2.Dopplergram.fits
    OK
[8/1906] hmi.v_45s.20240703_000600_TAI.2.Dopplergram.fits
    OK
[9/1906] hmi.v_45s.20240703_000645_TAI.2.Dopplergram.fits
    OK
[10/1906] hmi.v_45s.20240703_000730_TAI.2.Dopplergram.fits
    OK
[11/1906] hmi.v_45s.20240703_000815_TAI.2.Dopplergram.fits
    OK
[12/1906] hmi.v_45s.20240703_000900_TAI.2.Dopplergram.fits
    OK
[13/1906] hmi.v_45s.20240703_000945_TAI.2.Dopplergram.fits
    OK
[14/1906] hmi.v_45s.20240703_001030_TAI.2.Dopplergram.fits
    OK
[15/1906] hmi.v_45s.20240703_001115_TAI.2.Dopplergram.fits

In [3]:
from astropy.io import fits
from pathlib import Path
from collections import defaultdict

files = sorted(Path(".").glob("*.Dopplergram.fits"))

times = defaultdict(list)

for f in files:
    try:
        hdr = fits.getheader(f, 0)
        t = hdr.get("T_REC")
        times[t].append(f.name)
    except Exception as e:
        print("ERROR:", f.name, e)

print("Número de archivos:", len(files))
print("Número de T_REC únicos:", len(times))

duplicates = {
    t: names
    for t, names in times.items()
    if len(names) > 1
}

print("T_REC duplicados:", len(duplicates))

for t, names in duplicates.items():
    print("\n", t)
    for name in names:
        print("   ", name)

Número de archivos: 0
Número de T_REC únicos: 0
T_REC duplicados: 0
